# Toy model (3 reactions) with resource balance analysis (RBA)

Builds the toy model (TS, AAS, R) in the way of the RBA model of the ribosome composition paper (`RiboComp`, `model_functions.py`), and scans the growth rate while the proteome fraction of each enzyme is forced to values between 0 and 1. The scan has the same grid and the same definition of the proteome fraction as the COBRAk and GBA scans, so the results can be plotted together.

**How the RiboComp model is built, and how it is used here**

| RiboComp (`model_functions.py`) | Here |
|---|---|
| Columns `v...`: fluxes of the metabolic reactions. Columns `w...`: synthesis fluxes of the proteins (one column per enzyme) | `vTS`, `vAAS`, and `wTS`, `wAAS`, `wR` |
| Rows for the metabolites (`C`, `AA`, ...) and one row per enzyme (`matrix_uc`). The enzyme concentration is `c = w / mu` | rows `S`, `AA`, `TS`, `AAS`, `R` |
| Capacity rows `-mu*v + kcat*w >= 0`, each with its own slack column (`matrix_c`) | `capTS`, `capAAS`, `capR` |
| Ribosome capacity: `kel*c_R >= sum(n_i*w_i)`, with `c_R = v_AF / mu` | `capR`: `kR*w_R - mu*sum(w) >= 0`. In the toy the ribosome is a protein, so `c_R = w_R / mu` |
| Dry mass row `-omega_C*v_IC + mu*C = 0` with the constant column `C` | `-vTS + mu*C = 0` (mass units, `omega_C = 1`) |
| `efmtool` enumerates the elementary growth vectors at a fixed `mu`; bisection over `mu` finds the highest growth rate with a nonzero vector | A linear program (HiGHS in `scipy`) checks whether a feasible vector exists at a fixed `mu`; bisection over `mu` is the same |

**Why no efmtool.** The paper uses efmtool only to find out whether a growth vector exists at a given growth rate. A feasibility LP answers the same question, needs no Java, and is fast (it needs `scipy` 1.6 or newer, which includes the HiGHS solver). The elementary vector at the maximum growth rate is unique, so the LP also returns it. If you need all elementary growth vectors at a growth rate below the maximum, use efmtool.

**Units.** Mass units everywhere, as in the supplement of the toy model: fluxes in g/(gDW h), enzyme amounts in g/gDW, kcat in g product/(g protein h). The mass fraction matrix of the toy model (TS: Cext to S, AAS: S to AA, R: AA to protein) has all coefficients 1, so no molar masses are needed.

## 1. Imports and configuration

In [3]:
import csv
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import linprog

# GBA results of the toy model (also gives the conditions of the GBA simulation)
GBA_CSV = "../data/B_rev_protein_cost.csv"

# Optional: scan results of the COBRAk toy notebook, only used to compare and to plot
COBRAK_SCAN_JSON = "../data/toy_expression_scan_results__rev.json"

# True: the metabolite pools S and AA of the GBA optimum are part of the biomass (as in the COBRAk toy model).
# False: all biomass is protein (the RiboComp base model has no metabolite pools).
INCLUDE_METABOLITE_POOLS = True

# Grid of proteome fractions: the same as in the COBRAk toy notebook
PHI_GRID = np.unique(np.concatenate([np.linspace(0.0, 1.0, 51), np.logspace(-3, -1, 11)]))
PRINT_EVERY = 10   # print the progress after every PRINT_EVERY grid points

MU_SEARCH_MAX = 5.0    # 1/h, upper limit of the bisection
N_BISECTIONS = 45      # interval width 5 / 2^45
LP_OPTIONS = {"primal_feasibility_tolerance": 1e-10, "dual_feasibility_tolerance": 1e-10}

SCAN_JSON = "../data/toy_rba_scan_results.json"
FIGURE_FILE = "../data/toy_rba_scan.png"

## 2. Model data

In [4]:
# Forward turnover numbers of the toy model, g product / (g protein h)
KCAT = {"TS": 10.0, "AAS": 8.0, "R": 4.55}
ENZYMES = list(KCAT)

# Metabolite pools in the biomass at the GBA optimum (the row with the highest growth rate)
with open(GBA_CSV) as f:
    gba_rows = list(csv.DictReader(f))
gba_reaction = np.array([row["reaction"] for row in gba_rows])
gba = {key: np.array([float(row[key]) for row in gba_rows]) for key in gba_rows[0] if key != "reaction"}

i_opt = int(np.argmax(gba["mu"]))
density = sum(gba[f"c.{m}"][i_opt] for m in ("S", "AA", "P"))   # g/L
pools = {"S": gba["c.S"][i_opt] / density, "AA": gba["c.AA"][i_opt] / density}   # g/gDW
if not INCLUDE_METABOLITE_POOLS:
    pools = {"S": 0.0, "AA": 0.0}
protein_fraction = 1.0 - pools["S"] - pools["AA"]   # g protein / gDW

print(f"GBA optimum: growth rate {gba['mu'][i_opt]:.4f} 1/h, density {density:.1f} g/L")
print(f"biomass: S {pools['S']:.4f}, AA {pools['AA']:.4f}, protein {protein_fraction:.4f} g/gDW")

GBA optimum: growth rate 1.5170 1/h, density 340.0 g/L
biomass: S 0.1504, AA 0.0725, protein 0.7770 g/gDW


## 3. Matrices

The columns are the fluxes of the metabolic reactions `v`, the synthesis fluxes of the proteins `w` (one per enzyme), one slack variable per capacity constraint, and the constant column `C` (1 g of biomass).

* **Metabolite rows** (equalities): the metabolite is produced by the reactions, and consumed by the next reaction, by protein synthesis, and by the growth dilution of its pool (`mu` times the pool in the column `C`). One gram of amino acid gives one gram of protein, so all `n_i` are 1.
* **Capacity rows**: `mu*v <= kcat*w`, because the enzyme concentration is `c = w / mu`. The slack variable lets the capacity be larger than needed.
* **Ribosome row**: the ribosome makes all proteins, including itself: `kR*w_R >= mu*(wTS + wAAS + wR)`.
* **Dry mass row**: all mass enters as carbon, `vTS = mu`.
* **Forced protein fraction** (optional row): the fraction of enzyme `j` in the total protein is `w_j / sum(w)`.

`matrix_uc` has the enzyme rows and is used to compute the mass fractions: `(N v / mu)` in g/g.

In [5]:
columns_uc = ["vTS", "vAAS", "wTS", "wAAS", "wR"]
rows_uc = ["S", "AA", "TS", "AAS", "R"]
slack_columns = ["s_capTS", "s_capAAS", "s_capR"]
columns_c = columns_uc + slack_columns + ["C"]
rows_c = ["S", "AA", "capTS", "capAAS", "capR", "mass"]


def make_matrix_uc():
    """Stoichiometric matrix with a row for each metabolite and each enzyme (g/g units)."""
    return np.array([
        # vTS vAAS wTS wAAS wR
        [1, -1,  0,  0,  0],    # S:   made by TS, used by AAS
        [0,  1, -1, -1, -1],    # AA:  made by AAS, used for the synthesis of all proteins
        [0,  0,  1,  0,  0],    # TS:  made by wTS (concentration = wTS / mu)
        [0,  0,  0,  1,  0],    # AAS
        [0,  0,  0,  0,  1],    # R
    ], dtype=float)


def make_matrix_c(mu, pools, forced=None):
    """
    Matrix of all constraints at growth rate mu (equalities, with slack columns).
    forced = (enzyme, fraction): adds a row that fixes the fraction of the enzyme in the total protein.
    """
    kts, kaas, kr = KCAT["TS"], KCAT["AAS"], KCAT["R"]
    matrix = np.array([
        # vTS  vAAS   wTS    wAAS    wR        s_TS s_AAS s_R   C
        [ 1,   -1,     0,     0,      0,        0,   0,    0,   -mu * pools["S"]],    # S
        [ 0,    1,    -1,    -1,     -1,        0,   0,    0,   -mu * pools["AA"]],   # AA
        [-mu,   0,    kts,    0,      0,       -1,   0,    0,    0],                   # capTS:  mu*vTS <= kTS*wTS
        [ 0,  -mu,     0,   kaas,     0,        0,  -1,    0,    0],                   # capAAS: mu*vAAS <= kAAS*wAAS
        [ 0,    0,   -mu,    -mu,   kr - mu,    0,   0,   -1,    0],                   # capR:   mu*sum(w) <= kR*wR
        [-1,    0,     0,     0,      0,        0,   0,    0,    mu],                  # mass:   vTS = mu
    ], dtype=float)
    if forced is not None:
        enzyme, fraction = forced
        row = np.zeros(matrix.shape[1])
        for name in ENZYMES:
            row[columns_c.index(f"w{name}")] = -fraction
        row[columns_c.index(f"w{enzyme}")] += 1.0
        matrix = np.vstack([matrix, row])
    return matrix


def print_matrix(matrix, row_names, column_names):
    print(" " * 9 + "".join(f"{c:>9s}" for c in column_names))
    for name, row in zip(row_names, matrix):
        print(f"{name:9s}" + "".join(f"{x:9.3g}" for x in row))


print("matrix_uc")
print_matrix(make_matrix_uc(), rows_uc, columns_uc)
print()
print("matrix_c at mu = 1 1/h")
print_matrix(make_matrix_c(1.0, pools), rows_c, columns_c)

matrix_uc
               vTS     vAAS      wTS     wAAS       wR
S                1       -1        0        0        0
AA               0        1       -1       -1       -1
TS               0        0        1        0        0
AAS              0        0        0        1        0
R                0        0        0        0        1

matrix_c at mu = 1 1/h
               vTS     vAAS      wTS     wAAS       wR  s_capTS s_capAAS   s_capR        C
S                1       -1        0        0        0        0        0        0    -0.15
AA               0        1       -1       -1       -1        0        0        0  -0.0725
capTS           -1        0       10        0        0       -1        0        0        0
capAAS           0       -1        0        8        0        0       -1        0        0
capR             0        0       -1       -1     3.55        0        0       -1        0
mass            -1        0        0        0        0        0        0        0        1

## 4. Maximum growth rate

At a fixed `mu` all constraints are linear. The LP asks whether a nonnegative vector exists that satisfies them (the column `C` is fixed to 1). The highest growth rate with a feasible vector is found by bisection, as in the paper.

In [6]:
def feasible(mu, pools, forced=None):
    """Return the solution vector (without C) if a feasible vector exists at growth rate mu, otherwise None."""
    matrix = make_matrix_c(mu, pools, forced)
    a_eq = matrix[:, :-1]
    b_eq = -matrix[:, -1]          # column C is 1
    result = linprog(c=np.zeros(a_eq.shape[1]), A_eq=a_eq, b_eq=b_eq,
                     bounds=(0, None), method="highs", options=LP_OPTIONS)
    return result.x if result.status == 0 else None


def bisection_search_mu(pools, forced=None):
    """Highest growth rate at which a feasible vector exists, and the vector at that growth rate."""
    if feasible(MU_SEARCH_MAX, pools, forced) is not None:
        raise ValueError("MU_SEARCH_MAX is feasible; increase it")
    left, right = 0.0, MU_SEARCH_MAX
    best = None
    for _ in range(N_BISECTIONS):
        middle = 0.5 * (left + right)
        solution = feasible(middle, pools, forced) if middle > 0 else None
        if solution is None:
            right = middle
        else:
            left, best = middle, solution
    return left, best


mu_max, solution = bisection_search_mu(pools)
print(f"maximum growth rate (RBA): {mu_max:.6f} 1/h")

maximum growth rate (RBA): 2.061241 1/h


### Checks

1. The closed-form solution of the toy model. With the carbon flux `mu` and the fluxes of AAS and R given by mass conservation, the capacities add up to `mu_max = protein / (1/kTS + (AA + protein)/kAAS + protein/kR)`.
2. The mass fractions `(N v / mu)` from `matrix_uc`: the pools must reproduce the biomass composition, and the enzymes must add up to the protein fraction.

In [8]:
analytic = protein_fraction / (1 / KCAT["TS"]
                               + (pools["AA"] + protein_fraction) / KCAT["AAS"]
                               + protein_fraction / KCAT["R"])
assert abs(mu_max - analytic) < 1e-5, (mu_max, analytic)

## 5. Forced under- and overexpression

For each enzyme, its share `phi` of the total protein is fixed, and the highest growth rate is found with the same bisection. At `phi = 0` and `phi = 1` the growth rate is 0 (the reaction is essential, or no other enzyme is left), so these two points are set without solving.

The scan is checked against the closed form: the growth rate is the smaller of the limit set by the forced enzyme and the limit set by the other enzymes, which share the rest of the protein.

In [9]:
def analytic_mu(enzyme, phi):
    """Closed form of the growth rate with the fraction phi of the given enzyme."""
    flux_per_mu = {"TS": 1.0, "AAS": pools["AA"] + protein_fraction, "R": protein_fraction}   # flux / mu
    need = {e: flux_per_mu[e] / KCAT[e] for e in ENZYMES}                                      # protein / mu
    limit_forced = phi * protein_fraction / need[enzyme]
    limit_others = (1 - phi) * protein_fraction / sum(v for e, v in need.items() if e != enzyme)
    return min(limit_forced, limit_others)


scan = {e: {"phi": [], "rba": []} for e in ENZYMES}
n_points = len(PHI_GRID)
start_time = time.perf_counter()
max_deviation = 0.0

for enzyme_index, enzyme in enumerate(ENZYMES, start=1):
    print(f"[{enzyme_index}/{len(ENZYMES)}] enzyme {enzyme}: {n_points} grid points", flush=True)
    for point_index, phi in enumerate(PHI_GRID, start=1):
        phi = float(phi)
        if phi <= 0.0 or phi >= 1.0:
            mu = 0.0
        else:
            mu, _ = bisection_search_mu(pools, forced=(enzyme, phi))
            max_deviation = max(max_deviation, abs(mu - analytic_mu(enzyme, phi)))
        scan[enzyme]["phi"].append(phi)
        scan[enzyme]["rba"].append(mu)
        if point_index % PRINT_EVERY == 0 or point_index == n_points:
            print(f"    {enzyme}: point {point_index}/{n_points}, phi = {phi:.4f}, mu = {mu:.4f}, "
                  f"elapsed {time.perf_counter() - start_time:.1f} s", flush=True)
    print(f"    {enzyme} done", flush=True)

print(f"largest difference between the LP scan and the closed form: {max_deviation:.2e} 1/h")
assert max_deviation < 1e-4

with open(SCAN_JSON, "w") as f:
    json.dump(scan, f)

[1/3] enzyme TS: 61 grid points
    TS: point 10/61, phi = 0.0251, mu = 0.1952, elapsed 0.6 s
    TS: point 20/61, phi = 0.1800, mu = 1.3986, elapsed 1.2 s
    TS: point 30/61, phi = 0.3800, mu = 1.7394, elapsed 1.8 s
    TS: point 40/61, phi = 0.5800, mu = 1.1783, elapsed 2.4 s
    TS: point 50/61, phi = 0.7800, mu = 0.6172, elapsed 3.1 s
    TS: point 60/61, phi = 0.9800, mu = 0.0561, elapsed 4.3 s
    TS: point 61/61, phi = 1.0000, mu = 0.0000, elapsed 4.3 s
    TS done
[2/3] enzyme AAS: 61 grid points
    AAS: point 10/61, phi = 0.0251, mu = 0.1838, elapsed 5.5 s
    AAS: point 20/61, phi = 0.1800, mu = 1.3171, elapsed 7.6 s
    AAS: point 30/61, phi = 0.3800, mu = 1.7792, elapsed 9.8 s
    AAS: point 40/61, phi = 0.5800, mu = 1.2052, elapsed 13.0 s
    AAS: point 50/61, phi = 0.7800, mu = 0.6313, elapsed 15.0 s
    AAS: point 60/61, phi = 0.9800, mu = 0.0574, elapsed 17.4 s
    AAS: point 61/61, phi = 1.0000, mu = 0.0000, elapsed 17.4 s
    AAS done
[3/3] enzyme R: 61 grid points


## 6. Comparison with COBRAk and GBA

The RBA curves are plotted as points on top of the GBA curve and, if the file exists, the COBRAk results (enzyme-constrained LP and kinetic NLP). In this model, RBA and the enzyme-constrained LP use the same linear capacity constraints and the same biomass composition, so the two curves are expected to be identical.

In [10]:
cobrak = None
if os.path.exists(COBRAK_SCAN_JSON):
    with open(COBRAK_SCAN_JSON) as f:
        cobrak = json.load(f)
    if not INCLUDE_METABOLITE_POOLS:
        print("note: the COBRAk results include the metabolite pools, so differences are expected here")
    if all(np.allclose(scan[e]["phi"], cobrak[e]["phi"]) for e in ENZYMES):
        difference = max(abs(a - b) for e in ENZYMES for a, b in zip(scan[e]["rba"], cobrak[e]["linear"]))
        print(f"largest difference between RBA and the COBRAk enzyme-constrained LP: {difference:.2e} 1/h")
    else:
        print("the proteome fraction grids of RBA and COBRAk differ: the curves are plotted, but not compared")
else:
    print(f"{COBRAK_SCAN_JSON} not found: only RBA and GBA are plotted")

ValueError: operands could not be broadcast together with shapes (61,) (101,) 

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9, 5.5), sharex=True, sharey="row")
for col, enzyme in enumerate(ENZYMES):
    phi = np.array(scan[enzyme]["phi"])
    rba = np.array(scan[enzyme]["rba"])
    in_r = gba_reaction == enzyme
    order = np.argsort(gba["phi"][in_r])
    gba_phi = gba["phi"][in_r][order]
    gba_mu = gba["mu"][in_r][order]

    # Top row: growth rate. Bottom row: growth rate relative to the optimum of each method (shape)
    top, bottom = axes[0, col], axes[1, col]
    top.plot(gba_phi, gba_mu, color="tab:orange", zorder=1, label="GBA")
    bottom.plot(gba_phi, gba_mu / gba["mu"].max(), color="tab:orange", zorder=1)
    if cobrak is not None:
        cobrak_phi = np.array(cobrak[enzyme]["phi"])
        linear = np.array(cobrak[enzyme]["linear"], dtype=float)
        nonlinear = np.array([np.nan if v is None else v for v in cobrak[enzyme]["nonlinear"]], dtype=float)
        top.plot(cobrak_phi, linear, color="tab:blue", zorder=2, label="COBRA-k (LP)")
        top.plot(cobrak_phi, nonlinear, color="tab:green", zorder=2, label="COBRA-k (NLP)")
        bottom.plot(cobrak_phi, linear / np.nanmax(linear), color="tab:blue", zorder=2)
        bottom.plot(cobrak_phi, nonlinear / np.nanmax(nonlinear), color="tab:green", zorder=2)
    top.scatter(phi, rba, s=8, color="black", zorder=3, label="RBA")
    bottom.scatter(phi, rba / mu_max, s=8, color="black", zorder=3)
    top.set_title(enzyme, fontsize=10)

axes[1, 1].set_xlabel(r"Proteome fraction $\phi$")
axes[0, 0].set_ylabel(r"Growth rate $\mu$ [1/h]")
axes[1, 0].set_ylabel(r"Growth rate relative to optimum $\mu$/$\mu^*$")
axes[0, 0].legend(frameon=False, fontsize=8, loc="upper right")
for ax in axes.flat:
    ax.set_xlim(0, 1)
    ax.set_ylim(bottom=0)
fig.tight_layout()
fig.savefig(FIGURE_FILE, dpi=300)
plt.show()

## What is the same and what is different

* **RBA and the enzyme-constrained LP give the same curves here.** Both have linear capacity constraints (`v <= kcat * E`) and a fixed biomass composition. In RBA the protein pool is not a separate bound: the ribosome makes all proteins, and the dry mass row fixes the biomass to 1 g, so the proteins add up to the protein fraction of the biomass. The two formulations are the same linear problem. The curves have two straight segments: one where the forced enzyme limits growth and one where the other enzymes, which share the remaining protein, limit it.
* **Without metabolite pools** (`INCLUDE_METABOLITE_POOLS = False`) all biomass is protein, as in the RiboComp base model. The growth rate is higher, but the shape is the same.
* **No enzyme kinetics.** RBA has no saturation, so the growth rate is the same as in the COBRAk LP and higher than in the COBRAk NLP and in GBA. The rounded curves of GBA come from the dependence of the saturation on the metabolite concentrations, which RBA does not have.
* **Differences to RiboComp.** The toy has no nucleotides, rRNA and RNA polymerase, so there are no rRNA or rP rows with accumulation slacks. The ribosome is a protein, so its concentration is `w_R / mu` and there is no assembly flux.